# Qwen3-VL-4B-Instruct on SageMaker with Neuron (BYOC)

Deploy [Qwen3-VL-4B-Instruct](https://huggingface.co/Qwen/Qwen3-VL-4B-Instruct) to a SageMaker real-time endpoint on AWS Inferentia2 using a general-purpose BYOC (Bring Your Own Container) with vLLM + NxD Inference.

## Architecture

- **Container**: General-purpose BYOC based on the pre-built `pytorch-inference-vllm-neuronx` DLC (SDK 2.28, vLLM 0.13.0, NxDI 0.8.x)
- **Engine**: vLLM with NxD Inference (neuronx-distributed-inference) backend
- **Serving**: FastAPI proxy bridging SageMaker's `/ping` + `/invocations` to vLLM's native OpenAI API
- **Instance**: `ml.inf2.8xlarge` (2 NeuronCores, tp=2)
- **Host AMI**: `al2-ami-sagemaker-inference-neuron-2` (Neuron driver 2.19, required for ExtISA support)
- **Features**: Text-only chat, image+text VLM queries, OpenAI-compatible API

The container is **model-agnostic** -- it can deploy any model supported by vllm-neuron (Llama, Qwen, Pixtral, etc.) by changing environment variables. No code changes required.

## Prerequisites

1. **BYOC container built and pushed to ECR** (see `qwen3vl_4b_sagemaker_build_container.ipynb`)
2. **Model weights uploaded to S3** (see Step 2)
3. **AWS permissions**: SageMaker, ECR, S3, IAM roles
4. **boto3 >= 1.42** (required for `InferenceAmiVersion` parameter)

## Performance (ml.inf2.8xlarge)

| Workload | Server Throughput | E2E Latency |
|----------|------------------|-------------|
| Text-only (128 tokens) | ~48.7 tok/s | ~2.6s |
| Image+Text (235 tokens) | ~47.6 tok/s | ~4.9s |

## Why This Approach?

**Why BYOC**: No standard SageMaker DLC combines vLLM + NxD Inference + SDK 2.28 in a working configuration. The official DLCs crash on startup (`ModuleNotFoundError: No module named 'pkg_resources'`) and require multiple patches for VLM models.

**Why SDK 2.28**: SageMaker currently only offers inf2 and trn1 Neuron instances (no trn2). SDK 2.29 (NxD Inference 0.9) drops inf2/trn1 support, so SDK 2.28 is the latest compatible version.

**Why `InferenceAmiVersion`**: SageMaker's default Neuron host driver (v2.10) cannot load NEFFs with Extended ISA (ExtISA) instructions generated by the SDK 2.28 NxD Inference compiler. The `al2-ami-sagemaker-inference-neuron-2` AMI provides Neuron driver v2.19, which supports ExtISA.

## Step 1: Setup and Configuration

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import boto3
import json
import base64
import time
import os

sts_client = boto3.client('sts')
account_id = sts_client.get_caller_identity()['Account']
region = boto3.Session().region_name
sm_client = boto3.client('sagemaker', region_name=region)
sm_runtime = boto3.client('sagemaker-runtime', region_name=region)

print(f"AWS Account: {account_id}")
print(f"Region: {region}")
print(f"boto3 version: {boto3.__version__}")

# Verify boto3 version supports InferenceAmiVersion
assert tuple(int(x) for x in boto3.__version__.split('.')[:2]) >= (1, 42), \
    f"boto3 >= 1.42 required for InferenceAmiVersion. Got {boto3.__version__}. Run: pip install --upgrade boto3"

In [ ]:
# ============================================================
# USER CONFIGURATION - Edit these values
# ============================================================

# SageMaker execution role
ROLE_ARN = "arn:aws:iam::691320336001:role/service-role/AmazonSageMaker-ExecutionRole-20250910T103661"

# ECR container (general-purpose neuron-vllm-sagemaker image)
# Build with: qwen3vl_4b_sagemaker_build_container.ipynb
ECR_REPO = "neuron-vllm-sagemaker"
ECR_TAG = "sdk2.28-v8"
IMAGE_URI = f"{account_id}.dkr.ecr.{region}.amazonaws.com/{ECR_REPO}:{ECR_TAG}"

# S3 path to model weights (HuggingFace format, not tar.gz)
# Must contain: config.json, tokenizer files, safetensors weights
S3_BUCKET = f"sagemaker-{region}-{account_id}"
S3_MODEL_PREFIX = "qwen3vl-neuron/model-weights-only"
MODEL_DATA_URI = f"s3://{S3_BUCKET}/{S3_MODEL_PREFIX}/"

# Instance configuration
INSTANCE_TYPE = "ml.inf2.8xlarge"  # 2 NeuronCores, tp=2

# CRITICAL: Use the updated Neuron host AMI with driver v2.19
# Without this, the default driver (v2.10) cannot load ExtISA NEFFs
INFERENCE_AMI_VERSION = "al2-ami-sagemaker-inference-neuron-2"

print(f"Container: {IMAGE_URI}")
print(f"Instance: {INSTANCE_TYPE}")
print(f"Model data: {MODEL_DATA_URI}")
print(f"Host AMI: {INFERENCE_AMI_VERSION}")

## Step 2: Upload Model Weights to S3

Download the model from HuggingFace and upload to S3. Skip this cell if weights are already in S3.

The model weights are ~8.3 GB (2 safetensors shards + tokenizer + config files).

In [ ]:
# Check if model weights already exist in S3
s3_client = boto3.client('s3')
try:
    resp = s3_client.list_objects_v2(Bucket=S3_BUCKET, Prefix=S3_MODEL_PREFIX, MaxKeys=5)
    count = resp.get('KeyCount', 0)
    if count > 0:
        print(f"Model weights already in S3 ({count}+ files). Skipping download.")
        print(f"  s3://{S3_BUCKET}/{S3_MODEL_PREFIX}/")
    else:
        print("No model weights found in S3. Run the next cell to download and upload.")
except Exception as e:
    print(f"Error checking S3: {e}")
    print("Run the next cell to download and upload model weights.")

In [ ]:
# Download from HuggingFace and upload to S3 (skip if already uploaded)
# This cell requires: pip install huggingface_hub

# from huggingface_hub import snapshot_download
# 
# MODEL_ID = "Qwen/Qwen3-VL-4B-Instruct"
# LOCAL_PATH = "/tmp/qwen3vl_model"
# 
# print(f"Downloading {MODEL_ID}...")
# snapshot_download(
#     repo_id=MODEL_ID,
#     local_dir=LOCAL_PATH,
#     ignore_patterns=["*.gguf", "*.md", ".gitattributes"]
# )
# 
# # Upload to S3
# import subprocess
# cmd = f"aws s3 sync {LOCAL_PATH} s3://{S3_BUCKET}/{S3_MODEL_PREFIX}/ --exclude '*.md' --exclude '.git*'"
# print(f"Uploading to S3...")
# subprocess.run(cmd, shell=True, check=True)
# print("Upload complete.")

## Step 3: Deploy SageMaker Endpoint

**Startup time**: ~15 minutes on first deployment (model download + Neuron compilation from scratch).

The container is configured entirely via environment variables:

| Variable | Description | Value |
|----------|------------|-------|
| `SM_MODEL_ID` | Model path or HuggingFace ID | `/opt/ml/model` |
| `SM_TP_DEGREE` | Tensor parallel degree | `2` (inf2.8xlarge has 2 cores) |
| `SM_MAX_MODEL_LEN` | Max sequence length | `4096` |
| `SM_MAX_NUM_SEQS` | Max concurrent sequences | `1` (required for Qwen3-VL on NxDI 0.8.x) |
| `SM_NEURON_CONFIG` | JSON neuron config override | See below |
| `NEURON_ON_DEVICE_SAMPLING_DISABLED` | Disable NKI sampling kernels | `1` (required for inf2) |

We use boto3 directly (instead of the SageMaker SDK) because the `InferenceAmiVersion` parameter is required and the SageMaker SDK's `Model.deploy()` does not expose it.

### Why `InferenceAmiVersion` is Required

The NxD Inference compiler (SDK 2.28) generates NEFFs containing Extended ISA (ExtISA) instructions via NKI kernel calls. SageMaker's default Neuron host driver (v2.10) cannot execute these instructions, causing `Allocation Failure` errors at model load time. The `al2-ami-sagemaker-inference-neuron-2` AMI provides driver v2.19, which supports ExtISA.

This affects ALL models deployed via vLLM + vllm_neuron + NxD Inference on SageMaker.

In [ ]:
timestamp = int(time.time())
model_name = f"qwen3vl-neuron-{timestamp}"
config_name = f"qwen3vl-neuron-config-{timestamp}"
endpoint_name = f"qwen3vl-neuron-{timestamp}"

# Container environment variables
environment = {
    'SM_MODEL_ID': '/opt/ml/model',
    'SM_TP_DEGREE': '2',
    'SM_MAX_MODEL_LEN': '4096',
    'SM_MAX_NUM_SEQS': '1',  # Must be 1 for Qwen3-VL on NxDI 0.8.x
    'SM_NEURON_CONFIG': json.dumps({
        'text_neuron_config': {'on_device_sampling_config': None},
        'vision_neuron_config': {},
    }),
    'NEURON_ON_DEVICE_SAMPLING_DISABLED': '1',
}

# Step 3a: Create SageMaker Model
print(f"Creating model: {model_name}")
sm_client.create_model(
    ModelName=model_name,
    PrimaryContainer={
        'Image': IMAGE_URI,
        'Environment': environment,
        'ModelDataSource': {
            'S3DataSource': {
                'S3Uri': MODEL_DATA_URI,
                'S3DataType': 'S3Prefix',
                'CompressionType': 'None'
            }
        }
    },
    ExecutionRoleArn=ROLE_ARN
)
print(f"  Model created.")

# Step 3b: Create Endpoint Configuration with InferenceAmiVersion
print(f"Creating endpoint config: {config_name}")
sm_client.create_endpoint_config(
    EndpointConfigName=config_name,
    ProductionVariants=[{
        'VariantName': 'AllTraffic',
        'ModelName': model_name,
        'InstanceType': INSTANCE_TYPE,
        'InitialInstanceCount': 1,
        'InitialVariantWeight': 1.0,
        'ContainerStartupHealthCheckTimeoutInSeconds': 1800,  # 30 min for compile
        'ModelDataDownloadTimeoutInSeconds': 1800,             # 30 min for download
        'VolumeSizeInGB': 100,
        # CRITICAL: Use updated Neuron host AMI with driver v2.19
        'InferenceAmiVersion': INFERENCE_AMI_VERSION
    }]
)
print(f"  Endpoint config created with InferenceAmiVersion={INFERENCE_AMI_VERSION}")

# Step 3c: Create Endpoint
print(f"Creating endpoint: {endpoint_name}")
sm_client.create_endpoint(
    EndpointName=endpoint_name,
    EndpointConfigName=config_name
)
print(f"  Endpoint creation started.")
print(f"\nThis will take ~15 minutes (model download + Neuron compilation).")
print(f"Monitor progress in CloudWatch: /aws/sagemaker/Endpoints/{endpoint_name}")

In [ ]:
# Wait for endpoint to be InService
print(f"Waiting for endpoint {endpoint_name} to be InService...")
print(f"Start time: {time.strftime('%H:%M:%S')}")

while True:
    resp = sm_client.describe_endpoint(EndpointName=endpoint_name)
    status = resp['EndpointStatus']
    print(f"  [{time.strftime('%H:%M:%S')}] Status: {status}")

    if status == 'InService':
        print(f"\nEndpoint is InService!")
        break
    elif status == 'Failed':
        reason = resp.get('FailureReason', 'Unknown')
        print(f"\nEndpoint FAILED: {reason}")
        print(f"Check CloudWatch logs: /aws/sagemaker/Endpoints/{endpoint_name}")
        break
    elif status not in ('Creating', 'Updating'):
        print(f"\nUnexpected status: {status}")
        break

    time.sleep(60)

## Step 4: Connect to Endpoint

If reconnecting to an existing endpoint (e.g., after a kernel restart), set `endpoint_name` below.

In [ ]:
# Reconnect to existing endpoint (uncomment and set endpoint_name if needed)
# endpoint_name = "qwen3vl-neuron-XXXXXXXXXX"
# sm_runtime = boto3.client('sagemaker-runtime', region_name=region)

def invoke_endpoint(payload):
    """Send inference request to endpoint and return parsed response."""
    response = sm_runtime.invoke_endpoint(
        EndpointName=endpoint_name,
        ContentType='application/json',
        Body=json.dumps(payload)
    )
    return json.loads(response['Body'].read().decode())

print(f"Connected to endpoint: {endpoint_name}")

## Step 5: Health Check

In [ ]:
# Quick health check
health_payload = {
    "messages": [{"role": "user", "content": "Hello"}],
    "max_tokens": 5
}

t0 = time.time()
response = invoke_endpoint(health_payload)
latency = time.time() - t0

print(f"Status: OK")
print(f"Response: {response['choices'][0]['message']['content']}")
print(f"Latency: {latency:.2f}s")
print(f"Model: {response.get('model', 'unknown')}")

## Step 6: Text-Only Inference

In [ ]:
print("=" * 60)
print("TEXT-ONLY INFERENCE")
print("=" * 60)

payload = {
    "messages": [
        {
            "role": "user",
            "content": "Explain what a vision-language model is and how it works. Be concise."
        }
    ],
    "max_tokens": 256
}

t0 = time.time()
response = invoke_endpoint(payload)
latency = time.time() - t0

content = response['choices'][0]['message']['content']
usage = response.get('usage', {})

output_tokens = usage.get('completion_tokens', 0)
tok_per_sec = output_tokens / latency if latency > 0 else 0

print(f"\nResponse ({output_tokens} tokens, {latency:.2f}s, {tok_per_sec:.1f} tok/s):")
print(content[:2000])

## Step 7: Image + Text Inference

Images must be sent as base64-encoded data URIs. SageMaker endpoints cannot fetch external URLs directly.

Supported format: `data:image/jpeg;base64,/9j/...`

In [ ]:
import urllib.request

def image_url_to_base64(url):
    """Download image from URL and convert to base64 data URI."""
    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(req, timeout=30) as resp:
        img_bytes = resp.read()
    b64 = base64.b64encode(img_bytes).decode('utf-8')
    return f"data:image/jpeg;base64,{b64}"

# Load the Qwen demo image
print("Downloading sample image...")
IMAGE_URL = "https://qianwen-res.oss-cn-beijing.aliyuncs.com/Qwen-VL/assets/demo.jpeg"
image_b64 = image_url_to_base64(IMAGE_URL)
print(f"Image encoded: {len(image_b64)} chars")

In [ ]:
print("=" * 60)
print("IMAGE + TEXT INFERENCE: Image Description")
print("=" * 60)

payload = {
    "messages": [
        {
            "role": "user",
            "content": [
                {
                    "type": "image_url",
                    "image_url": {"url": image_b64}
                },
                {
                    "type": "text",
                    "text": "Describe this image in detail. What do you see?"
                }
            ]
        }
    ],
    "max_tokens": 256
}

t0 = time.time()
response = invoke_endpoint(payload)
latency = time.time() - t0

content = response['choices'][0]['message']['content']
usage = response.get('usage', {})
output_tokens = usage.get('completion_tokens', 0)
tok_per_sec = output_tokens / latency if latency > 0 else 0

print(f"\nResponse ({output_tokens} tokens, {latency:.2f}s, {tok_per_sec:.1f} tok/s):")
print(content[:2000])

In [ ]:
print("=" * 60)
print("IMAGE + TEXT INFERENCE: Visual Q&A")
print("=" * 60)

payload = {
    "messages": [
        {
            "role": "user",
            "content": [
                {
                    "type": "image_url",
                    "image_url": {"url": image_b64}
                },
                {
                    "type": "text",
                    "text": "What breed is the dog? What is the woman wearing?"
                }
            ]
        }
    ],
    "max_tokens": 128
}

t0 = time.time()
response = invoke_endpoint(payload)
latency = time.time() - t0

content = response['choices'][0]['message']['content']
usage = response.get('usage', {})
output_tokens = usage.get('completion_tokens', 0)
tok_per_sec = output_tokens / latency if latency > 0 else 0

print(f"\nResponse ({output_tokens} tokens, {latency:.2f}s, {tok_per_sec:.1f} tok/s):")
print(content[:2000])

## Step 8: Benchmark Suite

Run timed iterations for both text-only and image+text queries.
Reports mean, std, P50, P99 latency and throughput.

In [ ]:
import numpy as np

WARMUP_RUNS = 2
BENCHMARK_RUNS = 5

def run_benchmark(payload, warmup=WARMUP_RUNS, runs=BENCHMARK_RUNS, label=""):
    """Run warmup + timed benchmark iterations."""
    print(f"\n{'=' * 60}")
    print(f"BENCHMARK: {label}")
    print(f"  Warmup: {warmup}, Benchmark: {runs}")
    print(f"{'=' * 60}")

    # Warmup
    for i in range(warmup):
        t0 = time.time()
        r = invoke_endpoint(payload)
        lat = time.time() - t0
        toks = r.get('usage', {}).get('completion_tokens', 0)
        tps = toks / lat if lat > 0 else 0
        print(f"  Warmup {i+1}: {lat:.2f}s, {toks} tokens, {tps:.1f} tok/s")

    # Timed runs
    latencies = []
    throughputs = []
    token_counts = []

    for i in range(runs):
        t0 = time.time()
        r = invoke_endpoint(payload)
        lat = time.time() - t0
        toks = r.get('usage', {}).get('completion_tokens', 0)
        tps = toks / lat if lat > 0 else 0

        latencies.append(lat)
        throughputs.append(tps)
        token_counts.append(toks)
        print(f"  Run {i+1}: {lat:.2f}s, {toks} tokens, {tps:.1f} tok/s")

    lat = np.array(latencies)
    tp = np.array(throughputs)

    print(f"\n  --- Results ---")
    print(f"  E2E Latency: {lat.mean():.3f}s (std={lat.std():.3f}, P50={np.percentile(lat, 50):.3f}, P99={np.percentile(lat, 99):.3f})")
    print(f"  Throughput: {tp.mean():.1f} tok/s (std={tp.std():.1f}, min={tp.min():.1f}, max={tp.max():.1f})")
    print(f"  Tokens: {np.mean(token_counts):.0f} mean")

    return {
        "label": label,
        "latency_mean": float(lat.mean()),
        "latency_p50": float(np.percentile(lat, 50)),
        "throughput_mean": float(tp.mean()),
    }

print("Benchmark function ready.")

In [ ]:
# Text-only benchmark
text_payload = {
    "messages": [{"role": "user", "content": "Explain what a vision-language model is and how it works. Be concise."}],
    "max_tokens": 256
}

text_results = run_benchmark(text_payload, label="Text-Only")

In [ ]:
# Image+Text benchmark
image_payload = {
    "messages": [{
        "role": "user",
        "content": [
            {"type": "image_url", "image_url": {"url": image_b64}},
            {"type": "text", "text": "Describe this image in detail. What do you see?"}
        ]
    }],
    "max_tokens": 256
}

image_results = run_benchmark(image_payload, label="Image+Text")

In [ ]:
# Summary
print("\n" + "=" * 60)
print("BENCHMARK SUMMARY")
print("=" * 60)
print(f"\n{'Workload':<15} {'E2E Latency':>12} {'Throughput':>14}")
print("-" * 45)
for r in [text_results, image_results]:
    print(f"{r['label']:<15} {r['latency_p50']:>10.2f}s  {r['throughput_mean']:>10.1f} tok/s")

print(f"\nEndpoint: {endpoint_name}")
print(f"Instance: {INSTANCE_TYPE}")
print(f"Container: {IMAGE_URI}")
print(f"Host AMI: {INFERENCE_AMI_VERSION}")

## Step 9: Cleanup

**Important**: Delete the endpoint when done to stop charges.
Uncomment and run the cell below.

In [ ]:
# Uncomment to delete resources:

# sm_client.delete_endpoint(EndpointName=endpoint_name)
# print(f"Endpoint {endpoint_name} deleted")

# sm_client.delete_endpoint_config(EndpointConfigName=config_name)
# print(f"Endpoint config {config_name} deleted")

# sm_client.delete_model(ModelName=model_name)
# print(f"Model {model_name} deleted")

## Technical Notes

### Container Architecture

The BYOC container is based on the pre-built `pytorch-inference-vllm-neuronx` DLC which has vLLM 0.13.0, vllm-neuron 0.4.1, and NxD Inference 0.8.x pre-installed. On top of this base, the container adds:

1. **FastAPI SageMaker proxy** (`entrypoint.py`) -- bridges SageMaker's `/ping` + `/invocations` to vLLM's native `/health` + `/v1/chat/completions`
2. **Bug patches** applied at build time (see below)
3. **On-disk NxDI patches** applied at startup by `entrypoint.py` (for model-specific fixes)

Configuration is entirely via environment variables (`SM_MODEL_ID`, `SM_TP_DEGREE`, etc.) so the same container image works for any vllm-neuron supported model.

### Neuron Host Driver and `InferenceAmiVersion`

SageMaker's default Neuron host AMI ships driver v2.10, which predates Extended ISA (ExtISA) support. The NxD Inference compiler in SDK 2.28 generates NEFFs containing NKI kernel calls (e.g., `cumsum` for on-device sampling) that produce ExtISA instructions. Without the updated AMI, these NEFFs fail to load with:

```
tdrv_get_device_resource_va failed (ret=4) to get event semaphore vaddr
RuntimeError: Could not load the model status=4 message=Allocation Failure
```

The `InferenceAmiVersion` parameter in `ProductionVariant` selects an alternate host AMI:

| Value | Accelerator | Driver |
|-------|------------|--------|
| `al2-ami-sagemaker-inference-neuron-2` | Inferentia2 / Trainium | Neuron 2.19 |
| `al2-ami-sagemaker-inference-gpu-2` | GPU | NVIDIA 535 / CUDA 12.2 |

**This parameter requires boto3 >= 1.42.** The SageMaker Python SDK's `Model.deploy()` does not currently expose this parameter, which is why this notebook uses boto3 directly.

### Patches Applied in Container

**Build-time patches** (in Dockerfile):

1. **vllm_neuron Sampler import**: Fixes `TypeError: 'module' object is not callable` -- vllm_neuron 0.4.1 imports the sampler module instead of the Sampler class
2. **NxDI deepstack_vision_embeds forward**: NxDI 0.8.x has two bugs that drop the 25th argument (`deepstack_vision_embeds`) needed by Qwen3-VL. Fixed in `image_to_text_model_wrapper.py` and `model_wrapper.py`
3. **On-device sampling disabled**: `NEURON_ON_DEVICE_SAMPLING_DISABLED=1` prevents NKI cumsum kernels that generate PSUM instructions incompatible with inf2

**Startup patches** (in entrypoint.py, applied before launching vLLM):

4. **tie_word_embeddings**: Adds `update_state_dict_for_tied_weights` to `NeuronQwen3VLForCausalLM` so `embed_tokens.weight` is copied to `lm_head.weight`
5. **Vision encoder QKV split**: NxDI 0.8.x maps fused QKV weights but `fused_qkv=False` (default) expects separate q/k/v projections. The patch splits the fused weights.

### Qwen3-VL Specific Settings

- **`SM_MAX_NUM_SEQS=1`**: VLM compilation produces NEFFs with `batch_size=1`. Sending `batch_size>1` causes `ValueError: Input shape not found in input_shape_map`.
- **`on_device_sampling_config: None`**: Must be set in `text_neuron_config` to prevent NxDI from compiling NKI cumsum kernels, even when the env var is set.
- **`vision_neuron_config: {}`**: Required for VLM models; tells NxDI to use nested neuron_config format.

### Deploying Other Models

To deploy a different model (e.g., Llama 3.1 8B):

```python
environment = {
    'SM_MODEL_ID': '/opt/ml/model',  # or 'meta-llama/Llama-3.1-8B-Instruct'
    'SM_TP_DEGREE': '2',
    'SM_MAX_MODEL_LEN': '4096',
    'SM_MAX_NUM_SEQS': '4',  # Text-only models can use batch_size > 1
    'SM_NEURON_CONFIG': json.dumps({
        'on_device_sampling_config': None,  # Flat format for text-only models
    }),
    'NEURON_ON_DEVICE_SAMPLING_DISABLED': '1',
}
```

The entrypoint patches are model-aware and only apply when the relevant model class is detected.